# Improving a quick first fruit model — then running it on a chip built just for AI

The lab keeps a quick first attempt at a fruit classifier: trained for only two short passes, it gets a
fair number of photos wrong. This notebook downloads it, **fine-tunes** it further in front of you,
shows the before/after, and writes the three files you upload to the lab web app so your improved
version runs on the FPGA board.

An FPGA is a chip that can be reconfigured, at the hardware level, to do one job extremely efficiently —
here, "recognise what's in a photo."

| step | what happens | roughly how long |
|---|---|---|
| 1. dataset | download the photos we learn from (once; cached afterwards) | ~1 min first time |
| 2. before | download the quick first model and score it on held-out photos | ~5 s |
| 3. fine-tune | keep training it, watch it get better | ~20 s on a GPU |
| 4. export | convert it into a format the chip's toolchain understands | ~20 s |
| 5. after | same photos, fine-tuned model | ~5 s |
| 6. hand off | upload `fruit-demo/` in the web app | ~20 s to prepare, ~30 s to deliver |

Run the cells top to bottom (*Run All* works). Everything it writes lands next to this notebook.

In [ ]:
import importlib.util, subprocess, sys
# openvino: 2024.6 (the lab's) through 2026.4 export IR the lab's FPGA compiler accepts; 2026.x has Python 3.13/3.14 wheels
for mod, pkg in (("torch", "torch"), ("torchvision", "torchvision"), ("openvino", "openvino>=2024.6,<2026.5"),
                 ("huggingface_hub", "huggingface_hub"), ("matplotlib", "matplotlib"), ("PIL", "pillow")):
    if importlib.util.find_spec(mod) is None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=True)
        importlib.invalidate_caches()

import json, os, random, shutil, time, urllib.request, warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import openvino as ov
import torch
import torch.nn as nn
import torchvision
from PIL import Image
from torchvision import datasets, transforms

WORK = Path.cwd()                       # everything this notebook writes lives next to it
DATA, HELD, OUT = WORK / "data", WORK / "held-out", WORK / "fruit-demo"
LAB_URL = "http://localhost:8000"        # the lab web app; it also serves the quick first model's weights
EPOCHS = 8                              # enough to watch it improve
DEV = "cuda" if torch.cuda.is_available() else "cpu"
print(torch.cuda.get_device_name(0) if DEV == "cuda" else "no GPU: training on CPU (slow, minutes per epoch)")
print(f"torch {torch.__version__} | torchvision {torchvision.__version__} | openvino {ov.__version__}")

## 1. The photos it learns from

15 kinds of fruit, real photos from a public dataset (HF *Fruits-30*, Apache-2.0). 80 % go to training;
20 % are set aside as a validation set the model never trains on. Two validation photos per class are
also saved to `held-out/` — those 30 are our honest test set, used before and after fine-tuning.

In [ ]:
CLASSES = {  # dataset folder -> label
    "apples": "apple", "bananas": "banana", "cherries": "cherry", "grapes": "grape",
    "kiwifruit": "kiwi", "lemons": "lemon", "limes": "lime", "mangos": "mango",
    "oranges": "orange", "peaches": "peach", "pineapples": "pineapple",
    "pomegranates": "pomegranate", "raspberries": "raspberry", "strawberries": "strawberry",
    "watermelons": "watermelon",
}
labels = sorted(CLASSES.values())

if not (DATA / "val").exists():
    warnings.filterwarnings("ignore", module="tqdm")   # "IProgress not found" nag when ipywidgets is missing
    from huggingface_hub import snapshot_download
    print("downloading ~200 MB of photos (once; cached for later runs)")
    src = Path(snapshot_download("VinayHajare/Fruits-30", repo_type="dataset",
                                 allow_patterns=[f"FruitImageDataset/{d}/*" for d in CLASSES]))
    rng = random.Random(0)
    HELD.mkdir(exist_ok=True)
    for hf_dir, label in CLASSES.items():
        files = sorted((src / "FruitImageDataset" / hf_dir).glob("*.jpg"))
        rng.shuffle(files)
        n_val = max(2, len(files) // 5)
        for split, part in (("val", files[:n_val]), ("train", files[n_val:])):
            d = DATA / split / label
            d.mkdir(parents=True, exist_ok=True)
            for f in part:
                shutil.copyfile(f, d / f.name)
        for i, f in enumerate(files[:2]):           # held-out copies, downscaled (originals can be > 2 MB)
            img = Image.open(f).convert("RGB")
            img.thumbnail((800, 800))
            img.save(HELD / f"{label}_{i:02d}.jpg", quality=90)

for split in ("train", "val"):
    n = {c: len(list((DATA / split / c).glob("*.jpg"))) for c in labels}
    print(f"{split:5s} {sum(n.values()):4d} photos   " + "  ".join(f"{c} {k}" for c, k in n.items()))
held = sorted(HELD.glob("*.jpg"))
truth = lambda f: f.name.rsplit("_", 1)[0]
print(f"held-out {len(held)} photos in {HELD.name}/")

fig, axes = plt.subplots(3, 5, figsize=(12, 7.5))
for ax, c in zip(axes.flat, labels):
    ax.imshow(Image.open(next((DATA / "train" / c).glob("*.jpg"))).convert("RGB").resize((224, 224)))
    ax.set_title(c); ax.axis("off")
plt.tight_layout()

## 2. Before: the model the board is running now

The lab's model is ResNet-18 — a small, freely available network already trained on a million everyday photos
(ImageNet) — with a new 15-fruit "head" that was trained for just two quick passes over our photos, with
the rest of the network frozen. Good enough to be a fruit classifier, not good enough to trust. We
download exactly those weights from the lab page and score them on the 30 held-out photos. Green =
right, red = wrong. Note which ones it gets wrong; the board will get the same ones wrong.

In [ ]:
weights = torchvision.models.ResNet18_Weights.IMAGENET1K_V1
MEAN = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1)
STD = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1)


def center_square(img):
    # The board looks at the centre square of the camera frame (what the web page shows).
    s = min(img.size)
    l, t = (img.width - s) // 2, (img.height - s) // 2
    return img.crop((l, t, l + s, t + s))


def load_deploy_image(path) -> np.ndarray:
    # Exactly what the board feeds the model: RGB, centre square, 224x224 resize, float 0..255, NCHW.
    img = center_square(Image.open(path).convert("RGB")).resize((224, 224), Image.BILINEAR)
    return np.asarray(img, dtype=np.float32).transpose(2, 0, 1)[None]


def show_grid(guesses, title):
    # guesses: {file -> (label text, prob, hit)}
    fig, axes = plt.subplots(5, 6, figsize=(13, 11))
    for ax, f in zip(axes.flat, held):
        g, p, hit = guesses[f]
        ax.imshow(Image.open(f).convert("RGB").resize((224, 224)))
        ax.set_title(f"{g} {p:.0%}", color="green" if hit else "red", fontsize=9)
        ax.axis("off")
    fig.suptitle(title, fontsize=14)
    plt.tight_layout()


def resnet18(n_classes):
    m = torchvision.models.resnet18(weights=weights)
    m.avgpool = nn.AvgPool2d(7, stride=1)   # FPGA primitive; identical to adaptive pool at 224x224
    m.fc = nn.Linear(512, n_classes)
    return m


def score_held_out(net):
    net = net.eval()
    out = {}
    with torch.no_grad():
        for f in held:
            x = (torch.from_numpy(load_deploy_image(f)) - MEAN * 255) / (STD * 255)
            p = net(x).softmax(1)[0]
            guess = labels[int(p.argmax())]
            out[f] = (guess, float(p.max()), guess == truth(f))
    return out


if not (WORK / "fruit-v1.pt").exists():
    try:  # via a .part file, so a failed download never leaves a broken fruit-v1.pt behind
        with urllib.request.urlopen(f"{LAB_URL}/fruit-v1.pt", timeout=15) as r, open(WORK / "fruit-v1.pt.part", "wb") as f:
            shutil.copyfileobj(r, f)
    except OSError as e:
        raise SystemExit(f"Can't reach the lab web app at {LAB_URL} ({e}).\n"
                         f"In the lab web app, Advanced -> Your own model -> download fruit-v1.pt, save it into {WORK}, "
                         f"and run this cell again.") from None
    (WORK / "fruit-v1.pt.part").replace(WORK / "fruit-v1.pt")
ckpt = torch.load(WORK / "fruit-v1.pt", map_location="cpu", weights_only=True)
assert ckpt["classes"] == labels
model = resnet18(len(labels))
model.load_state_dict(ckpt["state_dict"])

before = score_held_out(model)
hits_before = sum(hit for _, _, hit in before.values())
print(f"the quick first model: {hits_before}/{len(held)} held-out photos right")
show_grid(before, f"BEFORE: the quick first model, {hits_before}/{len(held)}")

## 3. Fine-tuning, live

We pick up where the first model stopped: this time the *whole* network is allowed to adjust, gently, on
the training photos, for a few more passes (epochs). Watch the validation accuracy climb from where the
first model is now. Seconds, not days, because we start from a model that already sees.

(One detail that makes the network FPGA-friendly: its final averaging step is a fixed 7×7 pool, the
primitive the chip runs natively. With the standard one, that step and everything after it would silently
fall back to the board's general-purpose processor.)

In [ ]:
norm = transforms.Normalize(MEAN.flatten().tolist(), STD.flatten().tolist())
train_tf = transforms.Compose([transforms.RandomResizedCrop(224, scale=(0.5, 1.0)), transforms.RandomHorizontalFlip(),
                               transforms.ColorJitter(0.3, 0.3, 0.3, 0.05), transforms.ToTensor(), norm])
val_tf = transforms.Compose([transforms.Lambda(center_square), transforms.Resize((224, 224)), transforms.ToTensor(), norm])
train_ds = datasets.ImageFolder(DATA / "train", train_tf)
val_ds = datasets.ImageFolder(DATA / "val", val_tf)
assert train_ds.classes == labels == val_ds.classes
# Windows/macOS start workers with "spawn": they can't see functions defined in a notebook (center_square)
workers = min(4, os.cpu_count() or 1) if torch.multiprocessing.get_start_method() == "fork" else 0
train_dl = torch.utils.data.DataLoader(train_ds, 32, shuffle=True, num_workers=workers)
val_dl = torch.utils.data.DataLoader(val_ds, 64, num_workers=workers)

model = model.to(DEV)
backbone = [p for n, p in model.named_parameters() if not n.startswith("fc.")]
opt = torch.optim.AdamW([{"params": backbone, "lr": 1e-4}, {"params": model.fc.parameters(), "lr": 1e-3}],
                        weight_decay=1e-4)
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, EPOCHS * len(train_dl))
loss_fn = nn.CrossEntropyLoss(label_smoothing=0.1)
scaler = torch.amp.GradScaler(enabled=DEV == "cuda")


def evaluate():
    model.eval()
    hit = n = 0
    conf = torch.zeros(len(labels), len(labels), dtype=torch.int64)
    with torch.no_grad(), torch.autocast(DEV, enabled=DEV == "cuda"):
        for x, y in val_dl:
            pred = model(x.to(DEV)).float().cpu().argmax(1)
            hit += (pred == y).sum().item(); n += len(y)
            for t, p in zip(y, pred):
                conf[t, p] += 1
    return hit / n, conf


val_before, conf_before = evaluate()
print(f"epoch  0  (the first model as downloaded)  val accuracy {val_before:.1%}")
best, best_state, best_conf, history, t_start = val_before, {k: v.cpu().clone() for k, v in model.state_dict().items()}, conf_before, [{"epoch": 0, "loss": None, "val_top1": val_before}], time.time()
for epoch in range(1, EPOCHS + 1):
    model.train()
    t0, tot, cnt = time.time(), 0.0, 0
    for x, y in train_dl:
        x, y = x.to(DEV), y.to(DEV)
        with torch.autocast(DEV, enabled=DEV == "cuda"):
            loss = loss_fn(model(x), y)
        opt.zero_grad(set_to_none=True)
        scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); sched.step()
        tot += loss.item() * len(y); cnt += len(y)
    top1, conf = evaluate()
    history.append({"epoch": epoch, "loss": tot / cnt, "val_top1": top1})
    print(f"epoch {epoch:2d}  loss {tot / cnt:.3f}  val accuracy {top1:.1%}  {time.time() - t0:.0f} s")
    if top1 >= best:
        best, best_state, best_conf = top1, {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}, conf
print(f"validation accuracy {val_before:.1%} -> {best:.1%} in {time.time() - t_start:.0f} s")

In [ ]:
fig, (a1, a2) = plt.subplots(1, 2, figsize=(11, 3.5))
a1.plot([h["epoch"] for h in history], [h["val_top1"] for h in history], "o-")
a1.set(title="validation accuracy (epoch 0 = the first model)", xlabel="epoch", ylim=(0, 1))
a2.plot([h["epoch"] for h in history[1:]], [h["loss"] for h in history[1:]], "o-")
a2.set(title="training loss", xlabel="epoch")
plt.tight_layout()
print("confusion matrix on the validation set (rows = truth, cols = predicted); off-diagonal = mistakes\n")
print(" " * 12 + " ".join(f"{c[:3]:>3s}" for c in labels))
for c, row in zip(labels, best_conf.tolist()):
    print(f"{c:12s}" + " ".join(f"{v:3d}" for v in row))

## 4. Converting it for the chip

The trained model lives in PyTorch, which the board doesn't speak. This step converts it into a
hardware-neutral format (an OpenVINO *IR* — two files, `model.xml` + `model.bin`) that the FPGA toolchain
reads. The photo pre-processing and the final scoring are baked in, so the board just feeds in a picture
and reads out probabilities. Then we double-check the converted model agrees with the PyTorch one on
every validation photo.

In [ ]:
class Deploy(nn.Module):
    # Graph as exported: raw 0..255 RGB in, class probabilities out.
    def __init__(self, net):
        super().__init__()
        self.net = net
        self.register_buffer("mean", MEAN * 255.0)
        self.register_buffer("std", STD * 255.0)

    def forward(self, x):
        return torch.softmax(self.net((x - self.mean) / self.std), dim=1)


net = resnet18(len(labels))
net.load_state_dict(best_state)
deploy = Deploy(net).eval()

OUT.mkdir(exist_ok=True)
ir = ov.convert_model(deploy, example_input=torch.zeros(1, 3, 224, 224), input=[1, 3, 224, 224])
ir.inputs[0].get_tensor().set_names({"image"})
ir.outputs[0].get_tensor().set_names({"prob"})
ov.save_model(ir, OUT / "model.xml", compress_to_fp16=False)
(OUT / "categories.txt").write_text("".join(f"{c}\n" for c in labels))

cm = ov.Core().compile_model(str(OUT / "model.xml"), "CPU")
agree, n = 0, 0
with torch.no_grad():
    for f in sorted((DATA / "val").rglob("*.jpg")):
        x = load_deploy_image(f)
        p_ov, p_pt = cm(x)[0][0], deploy(torch.from_numpy(x))[0].numpy()
        assert np.abs(p_ov - p_pt).max() < 1e-3
        agree += int(p_ov.argmax() == p_pt.argmax()); n += 1
print(f"converted model agrees with PyTorch on {agree}/{n} validation photos")
for f in ("model.xml", "model.bin", "categories.txt"):
    print(f"{f:16s} {(OUT / f).stat().st_size / 1e6:7.2f} MB   {OUT.name}/{f}")

## 5. After: the same 30 photos, improved model

Same held-out photos as step 2, run through the converted model on this computer's CPU. Compare with the
"before" grid — this is the scorecard the board should reproduce once we upload.

In [ ]:
after = {}
for f in held:
    p = cm(load_deploy_image(f))[0][0]
    guess = labels[int(p.argmax())]
    after[f] = (guess, float(p.max()), guess == truth(f))
hits_after = sum(hit for _, _, hit in after.values())
print(f"before: {hits_before}/{len(held)}   after: {hits_after}/{len(held)}   (validation accuracy {val_before:.1%} -> {best:.1%})")
show_grid(after, f"AFTER fine-tuning: {hits_after}/{len(held)}")

## 6. Sending it to the board

In the lab web app, click **My model +** (it opens **Advanced → Your own model**), drop the three files
from `fruit-demo/` next to this notebook — `model.xml`, `model.bin`, `categories.txt` — name it
`fruit-demo` and click **Upload & deploy**. The lab checks the model is compatible, compiles it for the
FPGA (~20 s), then delivers it to the board over the network (~30 s).

The web app shows the board's own camera, classified live by the chip. Before uploading, the board runs
the lab's built-in `fruit` model, made with the full recipe from the repo's `training/README.md`: more
photos, practice on pictures that look like this camera and like a phone screen, and an extra `none`
answer for "no fruit here". Hold a fruit up to it, then again once `fruit-demo` is live: same board,
same camera, your model. Yours has no `none` answer, so with nothing in view it still names a fruit.
That extra answer is one of the steps the full recipe adds.

To run this notebook again from scratch, delete `data/`, `held-out/`, `fruit-demo/` and `fruit-v1.pt`.